In [80]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

file_path = Path("../Online Retail.xlsx")

if not file_path.exists():
    file_path = Path("Online Retail.xlsx")

df = pd.read_excel(file_path)
df_raw = df.copy()
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


<div dir="rtl" style="text-align: right;">

<h2>سؤال ۱: آیا نوع داده‌ها و قالب تاریخ برای آماده‌سازی مناسب است؟</h2>

<p>
بررسی نوع داده‌های دیتاست Online Retail نشان می‌دهد که بیشتر ستون‌ها از نوع داده مناسبی برخوردار هستند. ستون <code>InvoiceDate</code> از نوع <code>datetime64[ns]</code> است و تاریخ‌ها به‌درستی همراه با ساعت و دقیقه ثبت شده‌اند؛ بنابراین برای تحلیل‌های زمانی و استخراج ویژگی‌های مرتبط با رفتار خرید مشتریان مناسب است. ستون‌های <code>Quantity</code> و <code>UnitPrice</code> نیز به‌ترتیب از نوع عدد صحیح و اعشاری هستند و نیازی به تغییر نوع داده ندارند.
</p>

<p>
با این حال، ستون <code>CustomerID</code> از نوع <code>float64</code> است، در حالی که تمام شناسه‌های موجود دارای مقادیر صحیح هستند. به همین دلیل، نوع این ستون به <code>Int64</code> تغییر داده می‌شود تا ضمن حفظ ماهیت عددی شناسه‌ها، مقادیر گمشده نیز قابل نگهداری باشند. همچنین ستون‌های متنی به نوع <code>string</code> تبدیل می‌شوند تا عملیات پاک‌سازی و پردازش متن به‌صورت استاندارد انجام شود.
</p>

<h3>تصمیم برای آماده‌سازی</h3>

<p>
در مجموع، نوع داده‌ها و قالب تاریخ دیتاست مناسب هستند؛ اما برای آماده‌سازی بهتر، اصلاح نوع ستون <code>CustomerID</code>، استانداردسازی ستون‌های متنی و بررسی مقادیر گمشده ضروری است.
</p>

</div>

In [81]:
df_prep = df_raw.copy()

print("Data types before preparation:")
print(df_prep.dtypes)

df_prep["InvoiceDate"] = pd.to_datetime(
    df_prep["InvoiceDate"],
    format="%m/%d/%y %H:%M",
    errors="coerce"
)

print("\nInvalid dates:", df_prep["InvoiceDate"].isna().sum())
print("Date range:")
print(df_prep["InvoiceDate"].min())
print(df_prep["InvoiceDate"].max())

print("\nData types after preparation:")
print(df_prep.dtypes)

Data types before preparation:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object

Invalid dates: 0
Date range:
2010-12-01 08:26:00
2011-12-09 12:50:00

Data types after preparation:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object


In [82]:
duplicate_counts = (
    df.groupby(list(df.columns), dropna=False)
      .size()
      .reset_index(name="Count")
)

duplicate_counts = duplicate_counts[
    duplicate_counts["Count"] > 1
]

duplicate_counts = duplicate_counts.drop_duplicates().copy()

display(duplicate_counts)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Count
491,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom,2
499,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom,2
510,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom,2
511,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom,2
537,536412,21448,12 DAISY PEGS IN WOOD BOX,1,2010-12-01 11:49:00,1.65,17920.0,United Kingdom,2
...,...,...,...,...,...,...,...,...,...
534958,C572226,85066,CREAM SWEETHEART MINI CHEST,-1,2011-10-21 13:58:00,12.75,15321.0,United Kingdom,2
535288,C574095,22326,ROUND SNACK BOXES SET OF4 WOODLAND,-1,2011-11-03 09:54:00,2.95,12674.0,France,2
535377,C574510,22360,GLASS JAR ENGLISH CONFECTIONERY,-1,2011-11-04 13:25:00,2.95,15110.0,United Kingdom,2
535661,C575940,23309,SET OF 60 I LOVE LONDON CAKE CASES,-24,2011-11-13 11:38:00,0.55,17838.0,United Kingdom,2


<div dir="rtl" style="text-align: right;">

<h2>سؤال ۲: با داده‌های گمشده و ردیف‌های تکراری چگونه برخورد کنیم؟</h2>

<p>
بررسی مقادیر گمشده در دیتاست نشان می‌دهد که از مجموع 541,909 ردیف، ستون <code>CustomerID</code> دارای 135,080 مقدار گمشده، معادل 24.93 درصد، و ستون <code>Description</code> دارای 1,454 مقدار گمشده، معادل 0.27 درصد است. سایر ستون‌ها مقدار گمشده ندارند.
</p>

<h3>۱. نحوه برخورد با مقادیر گمشده</h3>

<table dir="rtl" style="width:100%; border-collapse:collapse; text-align:center;">
<thead>
<tr>
<th style="border:1px solid #999; padding:6px;">ستون</th>
<th style="border:1px solid #999; padding:6px;">تعداد مقادیر گمشده</th>
<th style="border:1px solid #999; padding:6px;">درصد</th>
<th style="border:1px solid #999; padding:6px;">روش پیشنهادی</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #999; padding:6px;"><code>CustomerID</code></td>
<td style="border:1px solid #999; padding:6px;">135,080</td>
<td style="border:1px solid #999; padding:6px;">24.93%</td>
<td style="border:1px solid #999; padding:6px;">حفظ مقادیر گمشده و عدم جایگزینی حدسی</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:6px;"><code>Description</code></td>
<td style="border:1px solid #999; padding:6px;">1,454</td>
<td style="border:1px solid #999; padding:6px;">0.27%</td>
<td style="border:1px solid #999; padding:6px;">بررسی و تکمیل از طریق StockCode در صورت امکان</td>
</tr>
</tbody>
</table>

<p>
<strong>ستون CustomerID:</strong> از آنجا که این ستون شناسه مشتریان را مشخص می‌کند، جایگزینی مقادیر گمشده با میانگین، میانه یا یک شناسه دلخواه منطقی نیست؛ زیرا ممکن است اطلاعات نادرستی درباره هویت مشتریان ایجاد کند. بنابراین مقادیر گمشده حفظ می‌شوند و در تحلیل‌هایی که به شناسایی مشتری نیاز دارند، ردیف‌های فاقد شناسه مشتری به‌صورت جداگانه مدیریت می‌شوند. با این حال، این ردیف‌ها همچنان می‌توانند در تحلیل فروش، محصولات و کشورهای مختلف مورد استفاده قرار گیرند.
</p>

<p>
<strong>ستون Description:</strong> با توجه به اینکه تعداد مقادیر گمشده این ستون بسیار کم است، ابتدا می‌توان با استفاده از ستون <code>StockCode</code> بررسی کرد که آیا برای کد کالای موردنظر توضیحات معتبری در سایر ردیف‌ها وجود دارد یا خیر. در صورت وجود یک توضیح مشخص و معتبر، می‌توان آن را جایگزین کرد. در غیر این صورت، مقدار گمشده حفظ می‌شود یا برای تحلیل‌های متنی از مقدار «نامشخص» استفاده می‌شود.
</p>

<h3>۲. نحوه برخورد با ردیف‌های تکراری</h3>

<p>
بررسی دیتاست نشان می‌دهد که تعداد 5,268 ردیف کاملاً تکراری وجود دارد که معادل حدود 0.97 درصد کل داده‌ها است. این ردیف‌ها از نظر مقدار تمام ستون‌ها با یکدیگر یکسان هستند.
</p>

<p>
وجود ردیف‌های تکراری می‌تواند باعث شمارش چندباره یک تراکنش یا افزایش غیرواقعی تعداد فروش و مقادیر آماری شود. بنابراین، پس از بررسی و اطمینان از تکراری بودن کامل ردیف‌ها، حذف نسخه‌های اضافی آن‌ها پیشنهاد می‌شود؛ به‌طوری‌که تنها یک نسخه از هر ردیف باقی بماند.
</p>

<h3>۳. کد پیشنهادی برای مدیریت داده‌های گمشده و تکراری</h3>

<p>ابتدا تعداد مقادیر گمشده و ردیف‌های تکراری را بررسی می‌کنیم:</p>

<pre dir="ltr" style="text-align:left; white-space:pre-wrap;"><code># بررسی مقادیر گمشده
print(df.isnull().sum())

# بررسی تعداد ردیف‌های تکراری
print("Duplicate rows:", df.duplicated().sum())</code></pre>

<p>سپس ردیف‌های کاملاً تکراری را حذف می‌کنیم:</p>

<pre dir="ltr" style="text-align:left; white-space:pre-wrap;"><code># حذف ردیف‌های تکراری
df = df.drop_duplicates()

# بررسی مجدد تعداد ردیف‌های تکراری
print("Duplicate rows after cleaning:", df.duplicated().sum())</code></pre>

<p>
مقادیر گمشده ستون <code>CustomerID</code> را در این مرحله حذف یا جایگزین نمی‌کنیم تا اطلاعات تراکنش‌ها برای سایر تحلیل‌ها حفظ شود. همچنین برای ستون <code>Description</code>، ابتدا امکان بازیابی توضیحات از طریق <code>StockCode</code> بررسی می‌شود.
</p>

<h3>۴. تصمیم نهایی برای آماده‌سازی</h3>

<p>
در مجموع، تصمیم گرفته می‌شود ردیف‌های کاملاً تکراری پس از بررسی حذف شوند، مقادیر گمشده ستون <code>CustomerID</code> بدون جایگزینی حفظ شوند و مقادیر گمشده ستون <code>Description</code> در صورت امکان با استفاده از کد کالا تکمیل شوند. این روش باعث می‌شود ضمن جلوگیری از حذف غیرضروری اطلاعات، کیفیت داده‌ها برای مراحل بعدی تحلیل و مدل‌سازی بهبود پیدا کند.
</p>

</div>

In [83]:
print("Missing values before preparation:")
print(df_prep.isna().sum())

print("\nExact duplicate rows:")
print(df_prep.duplicated().sum())

rows_before = len(df_prep)

df_prep = df_prep.drop_duplicates().copy()

print("\nRows removed as exact duplicates:", rows_before - len(df_prep))
print("Rows remaining:", len(df_prep))

missing_customer_rows = df_prep["CustomerID"].isna().sum()
print("\nRows with missing CustomerID:", missing_customer_rows)

df_prep = df_prep.dropna(subset=["CustomerID"]).copy()

df_prep["CustomerID"] = df_prep["CustomerID"].astype("int64")

print("Rows after excluding missing CustomerID:", len(df_prep))
print("Unique customers:", df_prep["CustomerID"].nunique())

Missing values before preparation:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Exact duplicate rows:
5268

Rows removed as exact duplicates: 5268
Rows remaining: 536641

Rows with missing CustomerID: 135037
Rows after excluding missing CustomerID: 401604
Unique customers: 4372


<div dir="rtl" style="text-align: right;">

<h2>سؤال ۳: کدام تراکنش‌ها برای تحلیل رفتار خرید معتبرند؟</h2>

<h3>۱. شناسایی فاکتورهای لغوشده</h3>

<p>
در این دیتاست، شماره فاکتورهایی که با حرف <code>C</code> شروع می‌شوند، نشان‌دهنده تراکنش‌های لغوشده یا برگشتی هستند. بررسی داده‌ها نشان می‌دهد:
</p>

<ul>
<li>تعداد ردیف‌های مربوط به فاکتورهای لغوشده: <strong>9,288</strong></li>
<li>تعداد فاکتورهای لغوشده منحصربه‌فرد: <strong>3,836</strong></li>
<li>سهم این ردیف‌ها از کل داده‌ها: حدود <strong>1.71 درصد</strong></li>
</ul>

<p>
این ردیف‌ها نباید در محاسبه خریدهای واقعی مشتریان به‌عنوان فروش مثبت در نظر گرفته شوند؛ اما برای تحلیل لغو سفارش و مرجوعی کالا ارزشمند هستند.
</p>

<h3>۲. بررسی تعداد کالاها (<code>Quantity</code>)</h3>

<p>
مقدار ستون <code>Quantity</code> نشان می‌دهد که چه تعداد از هر کالا در یک ردیف فاکتور ثبت شده است. بررسی این ستون نتایج زیر را نشان می‌دهد:
</p>

<ul>
<li>تعداد ردیف‌های دارای مقدار منفی: <strong>10,624</strong></li>
<li>تعداد ردیف‌های دارای مقدار صفر: <strong>0</strong></li>
<li>سایر ردیف‌ها دارای مقدار مثبت هستند.</li>
</ul>

<p>
مقادیر منفی معمولاً با مرجوعی کالا یا اصلاح تراکنش ارتباط دارند؛ بنابراین برای تحلیل خریدهای واقعی، ردیف‌های دارای <code>Quantity</code> منفی نباید به‌عنوان خرید مثبت محسوب شوند.
</p>

<h3>۳. بررسی قیمت واحد (<code>UnitPrice</code>)</h3>

<p>
در ستون <code>UnitPrice</code> نیز موارد زیر مشاهده شده است:
</p>

<ul>
<li>تعداد قیمت‌های منفی: <strong>2</strong></li>
<li>تعداد قیمت‌های صفر: <strong>2,515</strong></li>
<li>سایر مقادیر قیمت واحد مثبت هستند.</li>
</ul>

<p>
برای تحلیل رفتار خرید، بهتر است فقط ردیف‌هایی در نظر گرفته شوند که قیمت واحد آن‌ها بزرگ‌تر از صفر باشد؛ زیرا قیمت‌های صفر و منفی ممکن است مربوط به اقلام رایگان، اصلاحات یا ثبت‌های غیرمعمول باشند و ارزش خرید واقعی را به‌درستی نشان ندهند.
</p>

<h3>۴. بررسی شناسه مشتری (<code>CustomerID</code>)</h3>

<p>
برای تحلیل رفتار خرید مشتریان، وجود شناسه مشتری اهمیت زیادی دارد. در این دیتاست تعداد <strong>135,080</strong> ردیف فاقد <code>CustomerID</code> است.
</p>

<p>
این ردیف‌ها لزوماً تراکنش نامعتبر نیستند؛ اما نمی‌توان آن‌ها را با اطمینان به یک مشتری مشخص نسبت داد. بنابراین:
</p>

<ul>
<li>در تحلیل کلی فروش می‌توان این تراکنش‌ها را حفظ کرد.</li>
<li>در تحلیل‌های مشتری‌محور مانند تعداد خرید هر مشتری، وفاداری مشتریان و تحلیل RFM، فقط ردیف‌هایی استفاده می‌شوند که <code>CustomerID</code> معتبر دارند.</li>
</ul>

<h3>۵. معیارهای نهایی انتخاب تراکنش‌های معتبر</h3>

<p>
برای تحلیل رفتار خرید، معیارهای زیر در نظر گرفته می‌شوند:
</p>

<table dir="rtl" style="width:100%; border-collapse:collapse; text-align:right;">
<thead>
<tr>
<th style="border:1px solid #999; padding:8px;">معیار</th>
<th style="border:1px solid #999; padding:8px;">شرط معتبر بودن</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #999; padding:8px;">شماره فاکتور</td>
<td style="border:1px solid #999; padding:8px;">با حرف <code>C</code> شروع نشود</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:8px;">تعداد کالا (<code>Quantity</code>)</td>
<td style="border:1px solid #999; padding:8px;">بزرگ‌تر از صفر باشد</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:8px;">قیمت واحد (<code>UnitPrice</code>)</td>
<td style="border:1px solid #999; padding:8px;">بزرگ‌تر از صفر باشد</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:8px;">شناسه مشتری</td>
<td style="border:1px solid #999; padding:8px;">برای تحلیل مشتری‌محور خالی نباشد</td>
</tr>
</tbody>
</table>

<h3>تصمیم نهایی برای آماده‌سازی</h3>

<p>
در این پروژه، برای تحلیل رفتار خرید، فاکتورهای لغوشده، ردیف‌های دارای تعداد کالای منفی یا صفر و ردیف‌های دارای قیمت واحد منفی یا صفر از مجموعه تراکنش‌های خرید معتبر کنار گذاشته می‌شوند.
</p>

<p>
با این حال، ردیف‌های فاقد <code>CustomerID</code> صرفاً در تحلیل‌های مشتری‌محور حذف می‌شوند و برای سایر تحلیل‌های فروش قابل استفاده خواهند بود. همچنین بهتر است اطلاعات مربوط به فاکتورهای لغوشده و کالاهای مرجوعی در یک مجموعه جداگانه نگهداری شوند تا در صورت نیاز، امکان تحلیل لغو سفارش و مرجوعی نیز وجود داشته باشد.
</p>

<p>
<strong>نکته:</strong> تعداد نهایی تراکنش‌های معتبر باید پس از اعمال هم‌زمان تمام شرط‌ها محاسبه شود؛ زیرا ممکن است یک ردیف هم‌زمان چند ویژگی نامعتبر داشته باشد و نباید تعداد موارد نامعتبر را به‌صورت ساده با یکدیگر جمع کرد.
</p>

</div>

In [84]:
cancelled_mask = (
    df_prep["InvoiceNo"]
    .astype("string")
    .str.startswith("C", na=False)
)

valid_purchase_mask = (
    (~cancelled_mask)
    & (df_prep["Quantity"] > 0)
    & (df_prep["UnitPrice"] > 0)
    & (df_prep["InvoiceDate"].notna())
)

sales = df_prep.loc[valid_purchase_mask].copy()

sales["LineTotal"] = sales["Quantity"] * sales["UnitPrice"]

print("Rows in prepared customer data:", len(df_prep))
print("Valid purchase rows:", len(sales))
print("Unique customers in valid purchases:", sales["CustomerID"].nunique())

print("\nDate range of valid purchases:")
print(sales["InvoiceDate"].min())
print(sales["InvoiceDate"].max())

print("\nValid purchase sample:")
display(sales.head())

Rows in prepared customer data: 401604
Valid purchase rows: 392692
Unique customers in valid purchases: 4338

Date range of valid purchases:
2010-12-01 08:26:00
2011-12-09 12:50:00

Valid purchase sample:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,LineTotal
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


<div dir="rtl" style="text-align: right;">

<h2>سؤال ۴: آیا بعد از آماده‌سازی اولیه، مشکلات مهمی در داده باقی مانده است؟</h2>

<p>
بله. حتی پس از انجام آماده‌سازی اولیه، چند مسئله مهم در دیتاست <code>Online Retail</code> باقی می‌ماند که باید پیش از تحلیل نهایی و مدل‌سازی بررسی شوند. برخی از این موارد مربوط به کیفیت داده‌ها هستند و برخی دیگر به محدودیت‌های ذاتی مجموعه‌داده مربوط می‌شوند.
</p>

<h3>۱. باقی ماندن مقادیر گمشده در CustomerID</h3>

<p>
یکی از مهم‌ترین مشکلات دیتاست، وجود مقادیر گمشده در ستون <code>CustomerID</code> است. تعداد 135,080 ردیف، معادل 24.93 درصد کل داده‌ها، فاقد شناسه مشتری هستند.
</p>

<p>
از آنجا که جایگزینی این مقادیر با شناسه‌های حدسی می‌تواند باعث ایجاد اطلاعات نادرست شود، این مقادیر نباید به‌صورت تصادفی تکمیل شوند. بنابراین، در تحلیل‌های مشتری‌محور مانند تحلیل RFM و بررسی وفاداری مشتریان، فقط ردیف‌های دارای شناسه مشتری معتبر استفاده خواهند شد.
</p>

<h3>۲. وجود مقادیر پرت در Quantity و UnitPrice</h3>

<p>
بررسی آماری ستون‌های عددی نشان می‌دهد که برخی مقادیر بسیار بزرگ یا غیرعادی در داده‌ها وجود دارند:
</p>

<ul>
<li>در ستون <code>Quantity</code> بیشترین مقدار برابر با 80,995 و کمترین مقدار برابر با 80,995- است.</li>
<li>در ستون <code>UnitPrice</code> بیشترین مقدار برابر با 38,970 و کمترین مقدار برابر با 11,062.06- است.</li>
<li>تعداد مقادیر پرت بر اساس روش فاصله بین چارکی (IQR) در ستون <code>Quantity</code> برابر با 58,619 و در ستون <code>UnitPrice</code> برابر با 39,627 گزارش شده است.</li>
</ul>

<p>
این مقادیر ممکن است ناشی از سفارش‌های بسیار بزرگ، تراکنش‌های خاص، اصلاحات یا خطاهای ثبت اطلاعات باشند. بنابراین، حذف خودکار تمام مقادیر پرت مناسب نیست و باید پیش از حذف یا اصلاح، ماهیت آن‌ها بررسی شود.
</p>

<h3>۳. باقی ماندن تراکنش‌های لغوشده و مرجوعی</h3>

<p>
در دیتاست اصلی، 9,288 ردیف مربوط به فاکتورهای لغوشده وجود دارد. همچنین برخی ردیف‌ها دارای مقدار منفی در ستون <code>Quantity</code> هستند که می‌توانند نشان‌دهنده مرجوعی کالا یا اصلاح تراکنش باشند.
</p>

<p>
برای تحلیل خریدهای واقعی، این موارد باید از مجموعه تراکنش‌های معتبر جدا شوند؛ اما بهتر است اطلاعات آن‌ها به‌طور کامل حذف نشود، زیرا در تحلیل لغو سفارش‌ها و مرجوعی محصولات کاربرد دارند.
</p>

<h3>۴. احتمال وجود ناهماهنگی در اطلاعات محصولات</h3>

<p>
ستون‌های <code>StockCode</code> و <code>Description</code> اطلاعات مربوط به محصولات را نگهداری می‌کنند. ممکن است برای یک کد کالا توضیحات متفاوتی ثبت شده باشد یا برخی توضیحات دارای فاصله‌های اضافی و تفاوت‌های نوشتاری باشند.
</p>

<p>
بنابراین، لازم است ارتباط بین کد کالا و توضیحات آن بررسی و در صورت وجود ناهماهنگی، اطلاعات محصولات استانداردسازی شود.
</p>

<h3>۵. محدود بودن بازه زمانی داده‌ها</h3>

<p>
اطلاعات این دیتاست از تاریخ 1 دسامبر 2010 تا 9 دسامبر 2011 را پوشش می‌دهد. بنابراین، داده‌ها تقریباً یک سال را شامل می‌شوند؛ اما اطلاعات ماه دسامبر 2011 فقط مربوط به 9 روز ابتدایی ماه است.
</p>

<p>
این موضوع هنگام تحلیل روندهای ماهانه، فصلی و مقایسه فروش ماه‌های مختلف اهمیت دارد؛ زیرا مقایسه مستقیم دسامبر 2011 با ماه‌های کامل می‌تواند نتایج گمراه‌کننده‌ای ایجاد کند.
</p>

<h3>۶. تفاوت بین داده‌های فروش و داده‌های مشتریان</h3>

<p>
از آنجا که تعدادی از ردیف‌ها فاقد <code>CustomerID</code> هستند، مجموعه داده برای تحلیل کلی فروش کامل‌تر از تحلیل رفتار مشتریان است. بنابراین، نباید تمام تحلیل‌ها را روی یک مجموعه داده یکسان انجام داد.
</p>

<p>
بهتر است برای تحلیل فروش از تراکنش‌های معتبر بدون الزام به داشتن شناسه مشتری استفاده شود؛ اما برای تحلیل رفتار مشتریان، فقط تراکنش‌هایی در نظر گرفته شوند که شناسه مشتری معتبر دارند.
</p>

<h3>تصمیم نهایی برای آماده‌سازی</h3>

<p>
در مجموع، پس از آماده‌سازی اولیه هنوز برخی محدودیت‌ها و مشکلات در داده‌ها باقی می‌ماند. برای ادامه پروژه، اقدامات زیر در نظر گرفته می‌شوند:
</p>

<ul>
<li>حفظ مقادیر گمشده <code>CustomerID</code> و حذف آن‌ها فقط از تحلیل‌های مشتری‌محور.</li>
<li>بررسی و مدیریت مقادیر پرت در ستون‌های <code>Quantity</code> و <code>UnitPrice</code> بدون حذف خودکار آن‌ها.</li>
<li>جدا نگه داشتن فاکتورهای لغوشده و تراکنش‌های مرجوعی از خریدهای واقعی.</li>
<li>بررسی و استانداردسازی ارتباط بین <code>StockCode</code> و <code>Description</code>.</li>
<li>توجه به ناقص بودن اطلاعات ماه دسامبر 2011 در تحلیل‌های زمانی.</li>
<li>ساخت مجموعه داده‌های متناسب با نوع تحلیل، به‌ویژه تحلیل فروش و تحلیل رفتار مشتریان.</li>
</ul>

<p>
بنابراین، داده‌ها پس از پاک‌سازی اولیه برای ادامه تحلیل قابل استفاده هستند؛ اما پیش از مدل‌سازی باید محدودیت‌های باقی‌مانده در نظر گرفته شوند تا نتایج حاصل از تحلیل، قابل اعتمادتر و دقیق‌تر باشند.
</p>

</div>

In [85]:
quality_report = pd.Series({
    "Rows in sales": len(sales),
    "Missing CustomerID": sales["CustomerID"].isna().sum(),
    "Missing InvoiceDate": sales["InvoiceDate"].isna().sum(),
    "Exact duplicate rows": sales.duplicated().sum(),
    "Non-positive Quantity": (sales["Quantity"] <= 0).sum(),
    "Non-positive UnitPrice": (sales["UnitPrice"] <= 0).sum(),
    "Unique customers": sales["CustomerID"].nunique(),
    "Unique invoices": sales["InvoiceNo"].nunique()
})

print(quality_report)

Rows in sales             392692
Missing CustomerID             0
Missing InvoiceDate            0
Exact duplicate rows           0
Non-positive Quantity          0
Non-positive UnitPrice         0
Unique customers            4338
Unique invoices            18532
dtype: int64


<div dir="rtl" style="text-align: right;">

<h2>سؤال ۵: چگونه ردیف‌های کالایی را به اطلاعات قابل استفاده برای تحلیل مشتری تبدیل کنیم؟</h2>

<p>
در دیتاست <code>Online Retail</code> هر ردیف نشان‌دهنده خرید یک محصول در یک فاکتور است. بنابراین، ممکن است یک مشتری در چندین ردیف مختلف و حتی در چندین فاکتور ثبت شده باشد. برای تحلیل رفتار مشتریان، لازم است اطلاعات مربوط به هر مشتری از طریق تجمیع ردیف‌های کالایی به یک ردیف مستقل تبدیل شود.
</p>

<h3>۱. انتخاب تراکنش‌های مناسب</h3>

<p>
ابتدا باید فقط تراکنش‌های معتبر مربوط به خرید واقعی را انتخاب کنیم. برای این کار، فاکتورهای لغوشده، تعداد کالای منفی یا صفر و قیمت واحد منفی یا صفر کنار گذاشته می‌شوند. همچنین برای تحلیل مشتری‌محور، فقط ردیف‌هایی استفاده می‌شوند که دارای شناسه مشتری معتبر هستند.
</p>

<p>
در این مرحله، یک مجموعه داده مستقل برای تحلیل رفتار مشتریان ایجاد می‌کنیم تا داده‌های اصلی حفظ شوند.
</p>

<h3>۲. محاسبه ارزش هر ردیف خرید</h3>

<p>
برای محاسبه مبلغ خرید، یک ستون جدید به نام <code>TotalPrice</code> ایجاد می‌کنیم. این ستون از حاصل‌ضرب تعداد کالا در قیمت واحد به دست می‌آید:
</p>

<p style="text-align:center;">
<strong>TotalPrice = Quantity × UnitPrice</strong>
</p>

<p>
این ویژگی نشان می‌دهد که ارزش مالی هر ردیف کالایی چقدر بوده است.
</p>

<h3>۳. استخراج ویژگی‌های مشتریان</h3>

<p>
پس از آماده‌سازی تراکنش‌ها، اطلاعات هر مشتری را بر اساس <code>CustomerID</code> تجمیع می‌کنیم. ویژگی‌های زیر برای تحلیل رفتار مشتریان در نظر گرفته می‌شوند:
</p>

<table dir="rtl" style="width:100%; border-collapse:collapse; text-align:right;">
<thead>
<tr>
<th style="border:1px solid #999; padding:8px;">ویژگی</th>
<th style="border:1px solid #999; padding:8px;">روش محاسبه</th>
<th style="border:1px solid #999; padding:8px;">کاربرد</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #999; padding:8px;"><code>Frequency</code></td>
<td style="border:1px solid #999; padding:8px;">تعداد فاکتورهای منحصربه‌فرد مشتری</td>
<td style="border:1px solid #999; padding:8px;">اندازه‌گیری تعداد دفعات خرید</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:8px;"><code>Monetary</code></td>
<td style="border:1px solid #999; padding:8px;">مجموع مبلغ خریدهای مشتری</td>
<td style="border:1px solid #999; padding:8px;">اندازه‌گیری ارزش مالی مشتری</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:8px;"><code>Recency</code></td>
<td style="border:1px solid #999; padding:8px;">تعداد روزهای گذشته از آخرین خرید مشتری</td>
<td style="border:1px solid #999; padding:8px;">اندازه‌گیری میزان تازگی خرید</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:8px;"><code>TotalQuantity</code></td>
<td style="border:1px solid #999; padding:8px;">مجموع تعداد کالاهای خریداری‌شده</td>
<td style="border:1px solid #999; padding:8px;">بررسی حجم خرید مشتری</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:8px;"><code>UniqueProducts</code></td>
<td style="border:1px solid #999; padding:8px;">تعداد محصولات منحصربه‌فرد خریداری‌شده</td>
<td style="border:1px solid #999; padding:8px;">بررسی تنوع خرید مشتری</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:8px;"><code>AvgOrderValue</code></td>
<td style="border:1px solid #999; padding:8px;">مجموع مبلغ خرید تقسیم بر تعداد فاکتورها</td>
<td style="border:1px solid #999; padding:8px;">میانگین ارزش هر سفارش</td>
</tr>
</tbody>
</table>

<h3>۴. ایجاد دیتاست مشتری‌محور</h3>

<p>
برای تبدیل داده‌های کالایی به داده‌های مشتری‌محور، مراحل زیر را اجرا می‌کنیم:
</p>

<pre dir="ltr" style="text-align:left; white-space:pre-wrap;"><code>invoice_data = (
    sales.groupby(["CustomerID", "InvoiceNo"], as_index=False)
    .agg(
        InvoiceDate=("InvoiceDate", "max"),
        InvoiceAmount=("LineTotal", "sum"),
        TotalUnits=("Quantity", "sum"),
        ProductCount=("StockCode", "nunique")
    )
)

print("Invoice-level dataset shape:", invoice_data.shape)
display(invoice_data.head())

customer_summary = (
    invoice_data.groupby("CustomerID")
    .agg(
        InvoiceCount=("InvoiceNo", "nunique"),
        TotalSpend=("InvoiceAmount", "sum"),
        TotalUnits=("TotalUnits", "sum"),
        UniqueProducts=("ProductCount", "sum"),
        FirstPurchase=("InvoiceDate", "min"),
        LastPurchase=("InvoiceDate", "max")
    )
    .reset_index()
)

print("\nCustomer-level dataset shape:", customer_summary.shape)
display(customer_summary.head())</code></pre>

<h3>۵. بررسی نتیجه تجمیع اطلاعات</h3>

<p>
پس از اجرای کد، دیتاست جدیدی به نام <code>customer_summary</code> ایجاد می‌شود که هر ردیف آن مربوط به یک مشتری است. در این دیتاست، به‌جای اطلاعات جداگانه هر کالای خریداری‌شده، ویژگی‌های کلی رفتار خرید هر مشتری نگهداری می‌شود.
</p>

<p>
از آنجا که در دیتاست اصلی 4,372 شناسه مشتری منحصربه‌فرد وجود دارد، در صورت استفاده از تمام مشتریان دارای شناسه معتبر در تراکنش‌های انتخاب‌شده، تعداد ردیف‌های دیتاست جدید برابر با تعداد مشتریانی خواهد بود که حداقل یک خرید معتبر دارند.
</p>

<h3>۶. ارتباط با تحلیل RFM</h3>

<p>
یکی از مهم‌ترین کاربردهای این تبدیل، آماده‌سازی داده‌ها برای تحلیل RFM است. در این روش، مشتریان بر اساس سه معیار اصلی ارزیابی می‌شوند:
</p>

<ul>
<li><strong>Recency:</strong> مشتری چه مدت قبل آخرین خرید خود را انجام داده است؟</li>
<li><strong>Frequency:</strong> مشتری چند بار خرید کرده است؟</li>
<li><strong>Monetary:</strong> مشتری در مجموع چه مقدار هزینه کرده است؟</li>
</ul>

<p>
این سه ویژگی امکان شناسایی مشتریان وفادار، مشتریان ارزشمند، مشتریان کم‌فعال و مشتریانی را که مدتی است خرید نکرده‌اند فراهم می‌کنند.
</p>

<h3>تصمیم نهایی برای آماده‌سازی</h3>

<p>
در این مرحله، داده‌های کالایی پس از انتخاب تراکنش‌های معتبر، بر اساس شناسه مشتری تجمیع می‌شوند و یک دیتاست جدید با سطح تحلیل مشتری ایجاد می‌شود. ویژگی‌هایی مانند تعداد دفعات خرید، مجموع مبلغ خرید، فاصله زمانی از آخرین خرید، تعداد کالاهای خریداری‌شده و تنوع محصولات در این دیتاست محاسبه می‌شوند.
</p>

<p>
در نتیجه، داده‌ها از سطح تراکنش و محصول به سطح مشتری تبدیل می‌شوند و برای تحلیل رفتار خرید، بخش‌بندی مشتریان و اجرای روش RFM آماده خواهند شد.
</p>

</div>

In [86]:
invoice_data = (
    sales.groupby(["CustomerID", "InvoiceNo"], as_index=False)
    .agg(
        InvoiceDate=("InvoiceDate", "max"),
        InvoiceAmount=("LineTotal", "sum"),
        TotalUnits=("Quantity", "sum"),
        ProductCount=("StockCode", "nunique")
    )
)

print("Invoice-level dataset shape:", invoice_data.shape)
display(invoice_data.head())

customer_summary = (
    invoice_data.groupby("CustomerID")
    .agg(
        InvoiceCount=("InvoiceNo", "nunique"),
        TotalSpend=("InvoiceAmount", "sum"),
        TotalUnits=("TotalUnits", "sum"),
        UniqueProducts=("ProductCount", "sum"),
        FirstPurchase=("InvoiceDate", "min"),
        LastPurchase=("InvoiceDate", "max")
    )
    .reset_index()
)

print("\nCustomer-level dataset shape:", customer_summary.shape)
display(customer_summary.head())

Invoice-level dataset shape: (18532, 6)


,CustomerID,InvoiceNo,InvoiceDate,InvoiceAmount,TotalUnits,ProductCount
0,12346,541431,2011-01-18 10:01:00,77183.60,74215,1
1,12347,537626,2010-12-07 14:57:00,711.79,319,31
2,12347,542237,2011-01-26 14:30:00,475.39,315,29
3,12347,549222,2011-04-07 10:43:00,636.25,483,24
4,12347,556201,2011-06-09 13:01:00,382.52,196,18



Customer-level dataset shape: (4338, 7)


,CustomerID,InvoiceCount,TotalSpend,TotalUnits,UniqueProducts,FirstPurchase,LastPurchase
0,12346,1,77183.60,74215,1,2011-01-18 10:01:00,2011-01-18 10:01:00
1,12347,7,4310.00,2458,182,2010-12-07 14:57:00,2011-12-07 15:52:00
2,12348,4,1797.24,2341,27,2010-12-16 19:09:00,2011-09-25 13:13:00
3,12349,1,1757.55,631,73,2011-11-21 09:51:00,2011-11-21 09:51:00
4,12350,1,334.40,197,17,2011-02-02 16:01:00,2011-02-02 16:01:00


<div dir="rtl" style="text-align: right;">

<h2>سؤال ۶: چه ویژگی‌هایی از رفتار گذشته مشتری برای پیش‌بینی ریزش ساخته می‌شوند؟</h2>

<h3>۱. ویژگی‌های اصلی رفتار خرید مشتریان</h3>

<p>
برای پیش‌بینی ریزش، ویژگی‌های زیر از سوابق خرید مشتریان استخراج می‌شوند:
</p>

<table dir="rtl" style="width:100%; border-collapse:collapse; text-align:right;">
<thead>
<tr>
<th style="border:1px solid #999; padding:8px;">ویژگی</th>
<th style="border:1px solid #999; padding:8px;">روش محاسبه</th>
<th style="border:1px solid #999; padding:8px;">کاربرد در پیش‌بینی ریزش</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #999; padding:8px;"><code>Recency</code></td>
<td style="border:1px solid #999; padding:8px;">تعداد روزهای گذشته از آخرین خرید</td>
<td style="border:1px solid #999; padding:8px;">تشخیص مشتریانی که مدتی است خرید نکرده‌اند</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:8px;"><code>Frequency</code></td>
<td style="border:1px solid #999; padding:8px;">تعداد فاکتورهای خرید مشتری</td>
<td style="border:1px solid #999; padding:8px;">سنجش میزان تکرار خرید</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:8px;"><code>Monetary</code></td>
<td style="border:1px solid #999; padding:8px;">مجموع مبلغ خرید مشتری</td>
<td style="border:1px solid #999; padding:8px;">سنجش ارزش مالی مشتری</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:8px;"><code>AvgOrderValue</code></td>
<td style="border:1px solid #999; padding:8px;">میانگین مبلغ هر فاکتور</td>
<td style="border:1px solid #999; padding:8px;">بررسی متوسط هزینه مشتری در هر سفارش</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:8px;"><code>TotalQuantity</code></td>
<td style="border:1px solid #999; padding:8px;">مجموع تعداد کالاهای خریداری‌شده</td>
<td style="border:1px solid #999; padding:8px;">بررسی حجم خرید مشتری</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:8px;"><code>UniqueProducts</code></td>
<td style="border:1px solid #999; padding:8px;">تعداد محصولات منحصربه‌فرد خریداری‌شده</td>
<td style="border:1px solid #999; padding:8px;">سنجش تنوع خرید</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:8px;"><code>CustomerLifetime</code></td>
<td style="border:1px solid #999; padding:8px;">فاصله زمانی اولین تا آخرین خرید</td>
<td style="border:1px solid #999; padding:8px;">بررسی مدت فعالیت مشتری</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:8px;"><code>AvgDaysBetweenOrders</code></td>
<td style="border:1px solid #999; padding:8px;">میانگین فاصله زمانی بین خریدها</td>
<td style="border:1px solid #999; padding:8px;">شناسایی تغییر در الگوی خرید</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:8px;"><code>PurchaseFrequencyTrend</code></td>
<td style="border:1px solid #999; padding:8px;">تغییر تعداد خرید در بازه‌های زمانی مختلف</td>
<td style="border:1px solid #999; padding:8px;">تشخیص کاهش فعالیت مشتری</td>
</tr>
</tbody>
</table>

<h3>۲. ویژگی‌های زمانی و تغییر رفتار مشتری</h3>

<p>
علاوه بر ویژگی‌های اصلی، می‌توان با بررسی الگوی زمانی خرید، اطلاعات دقیق‌تری از رفتار مشتری استخراج کرد.
</p>

<p>برای مثال:</p>

<ul>
<li>تعداد خریدهای مشتری در ۳۰، ۶۰ و ۹۰ روز گذشته.</li>
<li>مبلغ خرید مشتری در بازه‌های زمانی مختلف.</li>
<li>تعداد ماه‌هایی که مشتری در آن‌ها خرید داشته است.</li>
<li>میانگین فاصله زمانی بین سفارش‌های متوالی.</li>
<li>میزان کاهش یا افزایش تعداد خرید در ماه‌های اخیر.</li>
</ul>

<p>
این ویژگی‌ها کمک می‌کنند مشتریانی که به‌تدریج فعالیت خرید خود را کاهش داده‌اند، شناسایی شوند.
</p>

<h3>۳. ساخت ویژگی‌های رفتاری با استفاده از Python</h3>

<p>
در کد زیر، ابتدا اطلاعات خرید هر مشتری در سطح فاکتور تجمیع می‌شود و سپس ویژگی‌های رفتاری از آن استخراج می‌شوند.
</p>

<h3>۴. تعریف برچسب ریزش مشتری (Churn Label)</h3>

<p>
در دیتاست <code>Online Retail</code> ستون مستقیمی به نام <code>Churn</code> وجود ندارد؛ بنابراین باید این ستون را بر اساس رفتار خرید مشتریان ایجاد کنیم.
</p>

<p>
یکی از روش‌های پیشنهادی، استفاده از قانون ۹۰ روز عدم خرید است. بر اساس این روش، اگر مشتری در یک بازه ۹۰ روزه هیچ خرید معتبری نداشته باشد، به‌عنوان مشتری ریزش‌کرده در نظر گرفته می‌شود.
</p>

<p>
برای جلوگیری از نشت اطلاعات (Data Leakage)، باید دو بازه زمانی مجزا تعریف کنیم:
</p>

<ol>
<li><strong>بازه مشاهده (Observation Window):</strong> اطلاعات خرید گذشته مشتری برای استخراج ویژگی‌ها.</li>
<li><strong>بازه پیش‌بینی (Prediction Window):</strong> اطلاعات خرید آینده برای تعیین وضعیت ریزش مشتری.</li>
</ol>

<p>
با توجه به اینکه آخرین تاریخ دیتاست ۹ دسامبر ۲۰۱۱ است، می‌توان از تاریخ ۱۰ سپتامبر تا ۹ دسامبر ۲۰۱۱ به‌عنوان بازه ۹۰ روزه ارزیابی استفاده کرد و اطلاعات قبل از ۱۰ سپتامبر را برای ساخت ویژگی‌ها در نظر گرفت.
</p>

<h3>تصمیم نهایی برای آماده‌سازی</h3>

<p>
در این پروژه، برای پیش‌بینی ریزش مشتریان، ویژگی‌هایی مانند تازگی آخرین خرید، تعداد دفعات خرید، مجموع مبلغ خرید، میانگین ارزش سفارش، تنوع محصولات و میانگین فاصله بین خریدها استخراج می‌شوند.
</p>

<p>
همچنین با استفاده از قانون ۹۰ روز عدم خرید، متغیر هدف <code>Churn</code> ساخته می‌شود. برای جلوگیری از نشت اطلاعات، ویژگی‌ها فقط از داده‌های گذشته استخراج می‌شوند و وضعیت ریزش مشتری بر اساس رفتار او در یک بازه زمانی آینده تعیین خواهد شد.
</p>

</div>

In [87]:
def build_customer_features(sales_data, reference_date, history_days=180):
    reference_date = pd.Timestamp(reference_date).normalize()
    history_start = reference_date - pd.Timedelta(days=history_days)

    # Use purchases strictly before the reference date
    history = sales_data.loc[
        (sales_data["InvoiceDate"] >= history_start)
        & (sales_data["InvoiceDate"] < reference_date)
    ].copy()

    if history.empty:
        return pd.DataFrame()

    invoice_level = (
        history.groupby(["CustomerID", "InvoiceNo"], as_index=False)
        .agg(
            InvoiceDate=("InvoiceDate", "max"),
            InvoiceAmount=("LineTotal", "sum"),
            TotalUnits=("Quantity", "sum"),
            ProductCount=("StockCode", "nunique")
        )
    )

    features = (
        history.groupby("CustomerID")
        .agg(
            FirstPurchase=("InvoiceDate", "min"),
            LastPurchase=("InvoiceDate", "max"),
            TotalSpend=("LineTotal", "sum"),
            TotalUnits=("Quantity", "sum"),
            UniqueProducts=("StockCode", "nunique"),
            ActiveDays=("InvoiceDate", lambda x: x.dt.normalize().nunique())
        )
    )

    invoice_stats = (
        invoice_level.groupby("CustomerID")
        .agg(
            Frequency=("InvoiceNo", "nunique"),
            AverageInvoiceValue=("InvoiceAmount", "mean")
        )
    )

    features = features.join(invoice_stats, how="left")

    features["Recency"] = (
        reference_date - features["LastPurchase"].dt.normalize()
    ).dt.days

    features["TenureDays"] = (
        features["LastPurchase"].dt.normalize()
        - features["FirstPurchase"].dt.normalize()
    ).dt.days

    # Average gap between consecutive invoices
    invoice_level = invoice_level.sort_values(
        ["CustomerID", "InvoiceDate"]
    )

    invoice_level["DaysBetweenPurchases"] = (
        invoice_level.groupby("CustomerID")["InvoiceDate"]
        .diff()
        .dt.total_seconds()
        / 86400
    )

    average_gap = (
        invoice_level.groupby("CustomerID")["DaysBetweenPurchases"]
        .mean()
        .rename("AverageDaysBetweenPurchases")
    )

    features = features.join(average_gap, how="left")
    features["ReferenceDate"] = reference_date

    return features.reset_index()
reference_date = "2011-06-01"

customer_features = build_customer_features(
    sales_data=sales,
    reference_date=reference_date,
    history_days=180
)

print("Dataset shape:", customer_features.shape)

print("\nColumns:")
print(customer_features.columns.tolist())

display(customer_features.head())

Dataset shape: (2659, 13)

Columns:
['CustomerID', 'FirstPurchase', 'LastPurchase', 'TotalSpend', 'TotalUnits', 'UniqueProducts', 'ActiveDays', 'Frequency', 'AverageInvoiceValue', 'Recency', 'TenureDays', 'AverageDaysBetweenPurchases', 'ReferenceDate']


,CustomerID,FirstPurchase,LastPurchase,TotalSpend,TotalUnits,UniqueProducts,ActiveDays,Frequency,AverageInvoiceValue,Recency,TenureDays,AverageDaysBetweenPurchases,ReferenceDate
0,12346,2011-01-18 10:01:00,2011-01-18 10:01:00,77183.60,74215,1,1,1,77183.600000,134,0,NaN,2011-06-01
1,12347,2010-12-07 14:57:00,2011-04-07 10:43:00,1823.43,1117,63,3,3,607.810000,55,121,60.411806,2011-06-01
2,12348,2010-12-16 19:09:00,2011-04-05 10:47:00,1487.24,2124,22,3,3,495.746667,57,110,54.825694,2011-06-01
3,12350,2011-02-02 16:01:00,2011-02-02 16:01:00,334.40,197,17,1,1,334.400000,119,0,NaN,2011-06-01
4,12352,2011-02-16 12:33:00,2011-03-22 16:08:00,1561.81,254,26,4,5,312.362000,71,34,8.537326,2011-06-01


<div dir="rtl" style="text-align: right;">

<h2>سؤال ۷: برچسب Churn چگونه بر اساس رفتار آینده مشتری ساخته می‌شود؟</h2>

<h3>۱. تعیین معیار ریزش مشتری</h3>

<p>معیار پیشنهادی برای تشخیص ریزش به این صورت است:</p>

<ul>
    <li>اگر مشتری در بازه ۹۰ روزه آینده هیچ خرید معتبری نداشته باشد، مقدار <code>Churn</code> برابر با ۱ خواهد بود.</li>
    <li>اگر مشتری حداقل یک خرید معتبر در این بازه داشته باشد، مقدار <code>Churn</code> برابر با ۰ خواهد بود.</li>
</ul>

<table dir="rtl" style="width:100%; border-collapse:collapse; text-align:right;">
    <thead>
        <tr>
            <th style="border:1px solid #999; padding:8px;">مقدار Churn</th>
            <th style="border:1px solid #999; padding:8px;">مفهوم</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="border:1px solid #999; padding:8px;"><code>1</code></td>
            <td style="border:1px solid #999; padding:8px;">مشتری در بازه آینده خریدی نداشته و ریزش‌کرده محسوب می‌شود.</td>
        </tr>
        <tr>
            <td style="border:1px solid #999; padding:8px;"><code>0</code></td>
            <td style="border:1px solid #999; padding:8px;">مشتری حداقل یک خرید در بازه آینده داشته و فعال محسوب می‌شود.</td>
        </tr>
    </tbody>
</table>

<h3>۲. تعیین بازه‌های زمانی</h3>

<p>با توجه به اینکه آخرین تاریخ موجود در دیتاست ۹ دسامبر ۲۰۱۱ است، بازه زمانی زیر را برای ساخت برچسب در نظر می‌گیریم:</p>

<table dir="rtl" style="width:100%; border-collapse:collapse; text-align:right;">
    <thead>
        <tr>
            <th style="border:1px solid #999; padding:8px;">نوع بازه</th>
            <th style="border:1px solid #999; padding:8px;">محدوده زمانی</th>
            <th style="border:1px solid #999; padding:8px;">کاربرد</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="border:1px solid #999; padding:8px;">Observation Window</td>
            <td style="border:1px solid #999; padding:8px;">تا ۹ سپتامبر ۲۰۱۱</td>
            <td style="border:1px solid #999; padding:8px;">استخراج ویژگی‌های رفتاری گذشته</td>
        </tr>
        <tr>
            <td style="border:1px solid #999; padding:8px;">Prediction Window</td>
            <td style="border:1px solid #999; padding:8px;">۱۰ سپتامبر تا ۹ دسامبر ۲۰۱۱</td>
            <td style="border:1px solid #999; padding:8px;">بررسی خریدهای آینده و تعیین برچسب Churn</td>
        </tr>
    </tbody>
</table>

<p>در این روش، اطلاعات بازه پیش‌بینی فقط برای تعیین برچسب استفاده می‌شود و نباید در محاسبه ویژگی‌های ورودی مدل دخالت داشته باشد.</p>

<h3>۳. ساخت برچسب Churn با استفاده از Python</h3>

<p>سپس مشتریانی را که در بازه مشاهده خرید داشته‌اند و مشتریانی را که در بازه پیش‌بینی خرید کرده‌اند، مشخص می‌کنیم.</p>

<h3>۴. بررسی توزیع برچسب‌ها</h3>

<p>پس از ساخت ستون <code>Churn</code>، باید تعداد مشتریان ریزش‌کرده و فعال را بررسی کنیم.</p>

<p>برای نمایش بهتر توزیع برچسب‌ها می‌توان از نمودار میله‌ای استفاده کرد.</p>

<h3>تصمیم نهایی برای آماده‌سازی</h3>

<p>در این پروژه، برچسب <code>Churn</code> بر اساس رفتار خرید مشتری در یک بازه ۹۰ روزه آینده ساخته می‌شود. مشتریانی که در این بازه هیچ خرید معتبری نداشته باشند، ریزش‌کرده و مشتریانی که حداقل یک خرید داشته باشند، فعال در نظر گرفته می‌شوند.</p>

<p>به این ترتیب، یک مسئله یادگیری نظارت‌شده (Supervised Learning) ایجاد می‌شود که در آن ویژگی‌های گذشته مشتری به‌عنوان متغیرهای ورودی و ستون <code>Churn</code> به‌عنوان متغیر هدف برای آموزش مدل استفاده خواهند شد.</p>

</div>

In [88]:
reference_dates = [
    "2011-06-01",
    "2011-07-01",
    "2011-08-01",
    "2011-09-01"
]

snapshots = []

for ref_date in reference_dates:
    ref_date = pd.Timestamp(ref_date)

    features = build_customer_features(
        sales_data=sales,
        reference_date=ref_date,
        history_days=180
    )

    if features.empty:
        continue

    future_end = ref_date + pd.Timedelta(days=90)

    future_customers = sales.loc[
        (sales["InvoiceDate"] >= ref_date)
        & (sales["InvoiceDate"] < future_end),
        "CustomerID"
    ].unique()

    features["Churn"] = (
        ~features["CustomerID"].isin(future_customers)
    ).astype(int)

    snapshots.append(features)

if snapshots:
    churn_data = pd.concat(snapshots, ignore_index=True)
else:
    churn_data = pd.DataFrame()

print("Prepared dataset shape:", churn_data.shape)

if not churn_data.empty:
    print("\nRows per reference date:")
    print(churn_data["ReferenceDate"].value_counts().sort_index())

    print("\nChurn label distribution:")
    print(churn_data["Churn"].value_counts())

    print("\nChurn rate by reference date:")
    print(churn_data.groupby("ReferenceDate")["Churn"].mean())

    print("\nSample:")
    print(churn_data.head())
    

Prepared dataset shape: (10926, 14)

Rows per reference date:
ReferenceDate
2011-06-01    2659
2011-07-01    2724
2011-08-01    2771
2011-09-01    2772
Name: count, dtype: int64

Churn label distribution:
Churn
0    5970
1    4956
Name: count, dtype: int64

Churn rate by reference date:
ReferenceDate
2011-06-01    0.499436
2011-07-01    0.482746
2011-08-01    0.441718
2011-09-01    0.392857
Name: Churn, dtype: float64

Sample:
   CustomerID       FirstPurchase        LastPurchase  TotalSpend  TotalUnits  \
0       12346 2011-01-18 10:01:00 2011-01-18 10:01:00    77183.60       74215   
1       12347 2010-12-07 14:57:00 2011-04-07 10:43:00     1823.43        1117   
2       12348 2010-12-16 19:09:00 2011-04-05 10:47:00     1487.24        2124   
3       12350 2011-02-02 16:01:00 2011-02-02 16:01:00      334.40         197   
4       12352 2011-02-16 12:33:00 2011-03-22 16:08:00     1561.81         254   

   UniqueProducts  ActiveDays  Frequency  AverageInvoiceValue  Recency  \
0       

<div dir="rtl" style="text-align: right;">

<h2>سؤال ۸: آیا در ساخت ویژگی‌ها و برچسب ریزش، نشت اطلاعات آینده رخ داده است؟</h2>

<h3>۱. بررسی نحوه تقسیم بازه‌های زمانی</h3>

<p>برای جلوگیری از نشت اطلاعات، داده‌ها به دو بازه مستقل تقسیم می‌شوند:</p>

<ul>
    <li><strong>Observation Window:</strong> تا پایان ۹ سپتامبر ۲۰۱۱؛ برای استخراج ویژگی‌های رفتاری مشتری.</li>
    <li><strong>Prediction Window:</strong> از ۱۰ سپتامبر ۲۰۱۱ به بعد؛ برای بررسی خریدهای آینده و تعیین برچسب <code>Churn</code>.</li>
</ul>

<p><strong>نکته مهم:</strong> اگر منظور دقیقاً یک بازه ۹۰ روزه از ابتدای ۱۰ سپتامبر باشد، این بازه تا پایان ۸ دسامبر ۲۰۱۱ ادامه دارد. بنابراین، برای تعریف دقیق ۹۰ روز باید تاریخ پایان را اصلاح کنیم و داده‌های ۹ دسامبر را وارد این بازه نکنیم.</p>

<h3>۲. مواردی که ممکن است باعث نشت اطلاعات شوند</h3>

<p>در ساخت ویژگی‌های رفتاری مشتری، باید به موارد زیر توجه کنیم:</p>

<table dir="rtl" style="width:100%; border-collapse:collapse; text-align:right;">
    <thead>
        <tr>
            <th style="border:1px solid #999; padding:8px;">ویژگی یا مرحله</th>
            <th style="border:1px solid #999; padding:8px;">روش صحیح</th>
            <th style="border:1px solid #999; padding:8px;">خطر نشت اطلاعات</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="border:1px solid #999; padding:8px;"><code>Recency</code></td>
            <td style="border:1px solid #999; padding:8px;">محاسبه بر اساس آخرین خرید تا پایان Observation Window</td>
            <td style="border:1px solid #999; padding:8px;">استفاده از آخرین خرید کل دیتاست</td>
        </tr>
        <tr>
            <td style="border:1px solid #999; padding:8px;"><code>Frequency</code></td>
            <td style="border:1px solid #999; padding:8px;">شمارش فاکتورهای مشتری در بازه مشاهده</td>
            <td style="border:1px solid #999; padding:8px;">شمارش تمام فاکتورها، حتی فاکتورهای آینده</td>
        </tr>
        <tr>
            <td style="border:1px solid #999; padding:8px;"><code>Monetary</code></td>
            <td style="border:1px solid #999; padding:8px;">محاسبه مجموع مبلغ خرید در بازه مشاهده</td>
            <td style="border:1px solid #999; padding:8px;">جمع مبلغ خریدهای بازه پیش‌بینی</td>
        </tr>
        <tr>
            <td style="border:1px solid #999; padding:8px;"><code>AvgOrderValue</code></td>
            <td style="border:1px solid #999; padding:8px;">محاسبه میانگین ارزش سفارش‌های گذشته</td>
            <td style="border:1px solid #999; padding:8px;">استفاده از ارزش تمام سفارش‌ها</td>
        </tr>
        <tr>
            <td style="border:1px solid #999; padding:8px;"><code>TotalQuantity</code></td>
            <td style="border:1px solid #999; padding:8px;">مجموع تعداد کالاهای خریداری‌شده در گذشته</td>
            <td style="border:1px solid #999; padding:8px;">استفاده از تعداد کالاهای آینده</td>
        </tr>
        <tr>
            <td style="border:1px solid #999; padding:8px;"><code>UniqueProducts</code></td>
            <td style="border:1px solid #999; padding:8px;">تعداد کالاهای متمایز خریداری‌شده در گذشته</td>
            <td style="border:1px solid #999; padding:8px;">استفاده از اطلاعات کل دوره</td>
        </tr>
        <tr>
            <td style="border:1px solid #999; padding:8px;"><code>Churn</code></td>
            <td style="border:1px solid #999; padding:8px;">تعیین وضعیت خرید مشتری در بازه آینده</td>
            <td style="border:1px solid #999; padding:8px;">استفاده از این ستون به‌عنوان ویژگی ورودی مدل</td>
        </tr>
    </tbody>
</table>

<h3>محاسبه ویژگی‌ها بدون نشت اطلاعات</h3>

<p>نکته بسیار مهم این است که ویژگی‌های رفتاری باید فقط از <code>observation_data</code> استخراج شوند، نه از کل <code>customer_data</code>.</p>

<p>آخرین تاریخ موجود در ستون <code>LastPurchase</code> باید قبل از شروع بازه پیش‌بینی، یعنی قبل از ۱۰ سپتامبر ۲۰۱۱ باشد.</p>

<p>همچنین هنگام آموزش مدل باید ستون <code>CustomerID</code> و ستون <code>Churn</code> از ویژگی‌های ورودی حذف شوند؛ زیرا <code>CustomerID</code> شناسه مشتری است و <code>Churn</code> متغیر هدف مدل محسوب می‌شود.</p>

<h3>تصمیم نهایی برای آماده‌سازی</h3>

<p>در این پروژه، برای جلوگیری از نشت اطلاعات آینده، ویژگی‌های رفتاری مشتری فقط با استفاده از داده‌های بازه مشاهده محاسبه می‌شوند و اطلاعات بازه پیش‌بینی صرفاً برای ساخت برچسب <code>Churn</code> مورد استفاده قرار می‌گیرند.</p>

<p>بنابراین، در صورت رعایت این جداسازی، نشت مستقیم اطلاعات آینده در ساخت ویژگی‌ها رخ نمی‌دهد. با این حال، اگر ویژگی‌ها از کل دیتاست محاسبه شوند و سپس برچسب ریزش به آن‌ها اضافه شود، احتمال نشت اطلاعات وجود خواهد داشت.</p>

</div>

In [89]:
if not churn_data.empty:

    excluded_from_features = [
        "CustomerID",
        "ReferenceDate",
        "FirstPurchase",
        "LastPurchase",
        "Churn"
    ]

    feature_columns = [
        column
        for column in churn_data.columns
        if column not in excluded_from_features
    ]

    X = churn_data[feature_columns].copy()
    y = churn_data["Churn"].copy()

    print("Feature columns:")
    print(feature_columns)

    print("\nTarget column:")
    print(y.name)

    print("\nFeatures contain target:", "Churn" in X.columns)
    print("Features contain CustomerID:", "CustomerID" in X.columns)
    print("Features contain ReferenceDate:", "ReferenceDate" in X.columns)

Feature columns:
['TotalSpend', 'TotalUnits', 'UniqueProducts', 'ActiveDays', 'Frequency', 'AverageInvoiceValue', 'Recency', 'TenureDays', 'AverageDaysBetweenPurchases']

Target column:
Churn

Features contain target: False
Features contain CustomerID: False
Features contain ReferenceDate: False


In [90]:
print("Feature columns:")
print(feature_columns)

print("\nchurn_data:")
display(churn_data)

suspicious_columns = [
    "Churn",
    "CustomerID",
    "ReferenceDate",
    "FirstPurchase",
    "LastPurchase"
]

print("\nPotential leakage columns in features:")
print([col for col in suspicious_columns if col in feature_columns])

Feature columns:
['TotalSpend', 'TotalUnits', 'UniqueProducts', 'ActiveDays', 'Frequency', 'AverageInvoiceValue', 'Recency', 'TenureDays', 'AverageDaysBetweenPurchases']

churn_data:


,CustomerID,FirstPurchase,LastPurchase,TotalSpend,TotalUnits,UniqueProducts,ActiveDays,Frequency,AverageInvoiceValue,Recency,TenureDays,AverageDaysBetweenPurchases,ReferenceDate,Churn
0,12346,2011-01-18 10:01:00,2011-01-18 10:01:00,77183.60,74215,1,1,1,77183.600000,134,0,NaN,2011-06-01,1
1,12347,2010-12-07 14:57:00,2011-04-07 10:43:00,1823.43,1117,63,3,3,607.810000,55,121,60.411806,2011-06-01,0
2,12348,2010-12-16 19:09:00,2011-04-05 10:47:00,1487.24,2124,22,3,3,495.746667,57,110,54.825694,2011-06-01,1
3,12350,2011-02-02 16:01:00,2011-02-02 16:01:00,334.40,197,17,1,1,334.400000,119,0,NaN,2011-06-01,1
4,12352,2011-02-16 12:33:00,2011-03-22 16:08:00,1561.81,254,26,4,5,312.362000,71,34,8.537326,2011-06-01,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10921,18280,2011-03-07 09:52:00,2011-03-07 09:52:00,180.60,45,10,1,1,180.600000,178,0,NaN,2011-09-01,1
10922,18281,2011-06-12 10:53:00,2011-06-12 10:53:00,80.82,54,7,1,1,80.820000,81,0,NaN,2011-09-01,1
10923,18282,2011-08-05 13:35:00,2011-08-05 13:35:00,100.21,75,7,1,1,100.210000,27,0,NaN,2011-09-01,1
10924,18283,2011-04-21 16:37:00,2011-07-14 13:20:00,637.23,439,124,5,6,106.205000,49,84,16.772639,2011-09-01,0



Potential leakage columns in features:
[]


<div dir="rtl" style="text-align: right;">

<h2>سؤال ۹: آیا داده‌های آماده‌شده برای ورود به مرحله Modeling مناسب‌اند؟</h2>

<h3>۱. بررسی وضعیت داده‌های آماده‌شده</h3>

<table dir="rtl" style="width:100%; border-collapse:collapse; text-align:right;">
    <thead>
        <tr>
            <th style="border:1px solid #999; padding:8px;">مورد</th>
            <th style="border:1px solid #999; padding:8px;">وضعیت مورد انتظار</th>
            <th style="border:1px solid #999; padding:8px;">تصمیم</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="border:1px solid #999; padding:8px;">نوع داده‌ها</td>
            <td style="border:1px solid #999; padding:8px;">ستون‌ها دارای نوع داده مناسب باشند</td>
            <td style="border:1px solid #999; padding:8px;">تبدیل شناسه‌ها به نوع متنی و <code>CustomerID</code> به عدد صحیح nullable</td>
        </tr>
        <tr>
            <td style="border:1px solid #999; padding:8px;">داده‌های گمشده</td>
            <td style="border:1px solid #999; padding:8px;">مقادیر گمشده مدیریت شده باشند</td>
            <td style="border:1px solid #999; padding:8px;">حذف یا جایگزینی فقط در صورت نیاز مدل</td>
        </tr>
        <tr>
            <td style="border:1px solid #999; padding:8px;">ردیف‌های تکراری</td>
            <td style="border:1px solid #999; padding:8px;">تکرارهای کاملاً یکسان حذف شده باشند</td>
            <td style="border:1px solid #999; padding:8px;">استفاده از <code>drop_duplicates()</code> پس از بررسی</td>
        </tr>
        <tr>
            <td style="border:1px solid #999; padding:8px;">تراکنش‌های نامعتبر</td>
            <td style="border:1px solid #999; padding:8px;">لغو سفارش‌ها و مقادیر غیرمعتبر جدا شده باشند</td>
            <td style="border:1px solid #999; padding:8px;">استفاده از خریدهای معتبر</td>
        </tr>
        <tr>
            <td style="border:1px solid #999; padding:8px;">ویژگی‌های رفتاری</td>
            <td style="border:1px solid #999; padding:8px;">ویژگی‌هایی مانند <code>Recency</code>، <code>Frequency</code> و <code>Monetary</code> ساخته شده باشند</td>
            <td style="border:1px solid #999; padding:8px;">محاسبه فقط از بازه مشاهده</td>
        </tr>
        <tr>
            <td style="border:1px solid #999; padding:8px;">برچسب هدف</td>
            <td style="border:1px solid #999; padding:8px;">ستون <code>Churn</code> ساخته شده باشد</td>
            <td style="border:1px solid #999; padding:8px;">استفاده از رفتار خرید در بازه آینده</td>
        </tr>
        <tr>
            <td style="border:1px solid #999; padding:8px;">نشت اطلاعات</td>
            <td style="border:1px solid #999; padding:8px;">اطلاعات آینده وارد ویژگی‌ها نشده باشد</td>
            <td style="border:1px solid #999; padding:8px;">جداسازی کامل بازه مشاهده و پیش‌بینی</td>
        </tr>
        <tr>
            <td style="border:1px solid #999; padding:8px;">مقادیر پرت</td>
            <td style="border:1px solid #999; padding:8px;">مقادیر غیرعادی بررسی شده باشند</td>
            <td style="border:1px solid #999; padding:8px;">بررسی و مدیریت بدون حذف خودکار</td>
        </tr>
    </tbody>
</table>

<h3>۲. بررسی ویژگی‌های ورودی مدل</h3>

<p>در این پروژه، ویژگی‌های زیر می‌توانند برای پیش‌بینی ریزش مشتری استفاده شوند:</p>

<table dir="rtl" style="width:100%; border-collapse:collapse; text-align:right;">
    <thead>
        <tr>
            <th style="border:1px solid #999; padding:8px;">نام ویژگی</th>
            <th style="border:1px solid #999; padding:8px;">مفهوم</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="border:1px solid #999; padding:8px;"><code>Recency</code></td>
            <td style="border:1px solid #999; padding:8px;">تعداد روزهای گذشته از آخرین خرید مشتری</td>
        </tr>
        <tr>
            <td style="border:1px solid #999; padding:8px;"><code>Frequency</code></td>
            <td style="border:1px solid #999; padding:8px;">تعداد فاکتورهای خرید مشتری</td>
        </tr>
        <tr>
            <td style="border:1px solid #999; padding:8px;"><code>Monetary</code></td>
            <td style="border:1px solid #999; padding:8px;">مجموع مبلغ خرید مشتری</td>
        </tr>
        <tr>
            <td style="border:1px solid #999; padding:8px;"><code>AvgOrderValue</code></td>
            <td style="border:1px solid #999; padding:8px;">میانگین ارزش سفارش‌های مشتری</td>
        </tr>
        <tr>
            <td style="border:1px solid #999; padding:8px;"><code>TotalQuantity</code></td>
            <td style="border:1px solid #999; padding:8px;">مجموع تعداد کالاهای خریداری‌شده</td>
        </tr>
        <tr>
            <td style="border:1px solid #999; padding:8px;"><code>UniqueProducts</code></td>
            <td style="border:1px solid #999; padding:8px;">تعداد کالاهای متمایز خریداری‌شده</td>
        </tr>
        <tr>
            <td style="border:1px solid #999; padding:8px;"><code>CustomerLifetime</code></td>
            <td style="border:1px solid #999; padding:8px;">فاصله زمانی بین اولین و آخرین خرید مشتری</td>
        </tr>
        <tr>
            <td style="border:1px solid #999; padding:8px;"><code>AvgDaysBetweenOrders</code></td>
            <td style="border:1px solid #999; padding:8px;">میانگین فاصله زمانی بین سفارش‌های مشتری</td>
        </tr>
    </tbody>
</table>

<p>این ویژگی‌ها رفتار گذشته مشتری را توصیف می‌کنند و می‌توانند به مدل کمک کنند تا الگوهای مرتبط با ریزش را شناسایی کند.</p>

<h3>۳. بررسی ستون هدف و حذف ستون‌های غیرضروری</h3>

<p>ستون <code>Churn</code> متغیر هدف مدل است و نباید در میان ویژگی‌های ورودی قرار بگیرد.</p>

<h3>۴. بررسی مقادیر گمشده و نامتناهی</h3>

<p>پیش از آموزش مدل باید مطمئن شویم که ویژگی‌های ورودی دارای مقادیر گمشده یا نامتناهی مدیریت‌نشده نیستند.</p>

<p>اگر در ویژگی‌هایی مانند <code>AvgDaysBetweenOrders</code> مقدار گمشده وجود داشته باشد، باید با توجه به مفهوم ویژگی و نوع مدل، روش مناسبی برای مدیریت آن انتخاب شود.</p>

<h3>۵. بررسی توزیع برچسب Churn</h3>

<p>یکی از بررسی‌های مهم پیش از مدل‌سازی، مشاهده تعداد مشتریان ریزش‌کرده و فعال است.</p>

<p>اگر تعداد مشتریان ریزش‌کرده و فعال اختلاف زیادی داشته باشد، با مسئله نامتوازن بودن کلاس‌ها (Class Imbalance) مواجه هستیم. در این صورت، باید هنگام مدل‌سازی از روش‌های ارزیابی و در صورت نیاز روش‌های مدیریت عدم توازن کلاس‌ها استفاده کنیم.</p>

<h3>۶. تقسیم داده‌ها برای آموزش و ارزیابی مدل</h3>

<p>پس از بررسی نهایی، می‌توان داده‌ها را به مجموعه آموزش و آزمون تقسیم کرد.</p>

<h3>تصمیم نهایی برای آماده‌سازی</h3>

<p>در مجموع، داده‌های پروژه پس از انجام مراحل Data Preparation از نظر ساختار و نوع ویژگی‌ها قابلیت ورود به مرحله Modeling را دارند؛ اما پیش از آموزش مدل باید موارد زیر تأیید شوند:</p>

<ul>
    <li>مقادیر گمشده و نامتناهی ویژگی‌ها مدیریت شده باشند.</li>
    <li>ستون‌های شناسه و متغیر هدف از ویژگی‌های ورودی حذف شده باشند.</li>
    <li>ویژگی‌ها فقط از داده‌های بازه مشاهده استخراج شده باشند.</li>
    <li>برچسب <code>Churn</code> فقط بر اساس رفتار آینده مشتری ساخته شده باشد.</li>
    <li>توزیع کلاس‌های ریزش و عدم ریزش بررسی شده باشد.</li>
    <li>داده‌ها به‌درستی به مجموعه آموزش و آزمون تقسیم شوند.</li>
</ul>

</div>

In [91]:
if churn_data.empty:
    print("No prepared snapshots were created.")
else:
    print("Dataset shape:", churn_data.shape)

    print("\nMissing values in each column:")
    print(churn_data.isna().sum())

    print("\nTarget values:")
    print(sorted(churn_data["Churn"].dropna().unique()))

    print("\nTarget distribution:")
    print(churn_data["Churn"].value_counts(dropna=False))

    print("\nTarget rate by reference date:")
    print(
        churn_data.groupby("ReferenceDate")["Churn"]
        .agg(["count", "mean"])
    )

    print("\nDuplicate customer-reference pairs:")
    print(
        churn_data.duplicated(
            subset=["CustomerID", "ReferenceDate"]
        ).sum()
    )

Dataset shape: (10926, 14)

Missing values in each column:
CustomerID                        0
FirstPurchase                     0
LastPurchase                      0
TotalSpend                        0
TotalUnits                        0
UniqueProducts                    0
ActiveDays                        0
Frequency                         0
AverageInvoiceValue               0
Recency                           0
TenureDays                        0
AverageDaysBetweenPurchases    5166
ReferenceDate                     0
Churn                             0
dtype: int64

Target values:
[np.int64(0), np.int64(1)]

Target distribution:
Churn
0    5970
1    4956
Name: count, dtype: int64

Target rate by reference date:
               count      mean
ReferenceDate                 
2011-06-01      2659  0.499436
2011-07-01      2724  0.482746
2011-08-01      2771  0.441718
2011-09-01      2772  0.392857

Duplicate customer-reference pairs:
0


<div dir="rtl" style="text-align: right;">

<h2>سؤال ۱۰: خروجی نهایی Data Preparation در چه قالبی ذخیره می‌شود؟</h2>

<h3>ساختار خروجی نهایی</h3>

<p>مجموعه‌داده نهایی شامل ویژگی‌های رفتاری مشتری و ستون هدف <code>Churn</code> خواهد بود.</p>

<table dir="rtl" style="width:100%; border-collapse:collapse; text-align:right;">
    <thead>
        <tr>
            <th style="border:1px solid #999; padding:8px;">ستون</th>
            <th style="border:1px solid #999; padding:8px;">نوع اطلاعات</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="border:1px solid #999; padding:8px;"><code>Recency</code></td>
            <td style="border:1px solid #999; padding:8px;">ویژگی رفتاری</td>
        </tr>
        <tr>
            <td style="border:1px solid #999; padding:8px;"><code>Frequency</code></td>
            <td style="border:1px solid #999; padding:8px;">ویژگی رفتاری</td>
        </tr>
        <tr>
            <td style="border:1px solid #999; padding:8px;"><code>Monetary</code></td>
            <td style="border:1px solid #999; padding:8px;">ویژگی رفتاری</td>
        </tr>
        <tr>
            <td style="border:1px solid #999; padding:8px;"><code>AvgOrderValue</code></td>
            <td style="border:1px solid #999; padding:8px;">ویژگی رفتاری</td>
        </tr>
        <tr>
            <td style="border:1px solid #999; padding:8px;"><code>TotalQuantity</code></td>
            <td style="border:1px solid #999; padding:8px;">ویژگی رفتاری</td>
        </tr>
        <tr>
            <td style="border:1px solid #999; padding:8px;"><code>UniqueProducts</code></td>
            <td style="border:1px solid #999; padding:8px;">ویژگی رفتاری</td>
        </tr>
        <tr>
            <td style="border:1px solid #999; padding:8px;"><code>CustomerLifetime</code></td>
            <td style="border:1px solid #999; padding:8px;">ویژگی رفتاری</td>
        </tr>
        <tr>
            <td style="border:1px solid #999; padding:8px;"><code>AvgDaysBetweenOrders</code></td>
            <td style="border:1px solid #999; padding:8px;">ویژگی رفتاری</td>
        </tr>
        <tr>
            <td style="border:1px solid #999; padding:8px;"><code>Churn</code></td>
            <td style="border:1px solid #999; padding:8px;">متغیر هدف</td>
        </tr>
    </tbody>
</table>

<h3>تصمیم نهایی برای آماده‌سازی</h3>

<p>در پایان مرحله Data Preparation، مجموعه‌داده نهایی با نام <code>final_churn_dataset</code> در قالب‌های CSV و Pickle ذخیره می‌شود. این مجموعه شامل ویژگی‌های استخراج‌شده از رفتار گذشته مشتریان و ستون هدف <code>Churn</code> است و برای ورود به مرحله Modeling مورد استفاده قرار می‌گیرد.</p>

<p>به این ترتیب، مرحله Data Preparation در چارچوب CRISP-DM تکمیل شده و داده‌ها برای آموزش و ارزیابی مدل‌های پیش‌بینی ریزش مشتری آماده خواهند بود.</p>

</div>

In [92]:
if not churn_data.empty:
    output_path = "Customer_Churn_Prepared.csv"

    churn_data.to_csv(
        output_path,
        index=False
    )

    print("Prepared dataset saved to:", output_path)
    print("Final shape:", churn_data.shape)
else:
    print("Nothing to save: churn_data is empty.")

Prepared dataset saved to: Customer_Churn_Prepared.csv
Final shape: (10926, 14)
